In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
DATA_DIR = Path('')
print(f'데이터 폴더 : {DATA_DIR.resolve()}')

데이터 폴더 : C:\bigdata2026\DATA_ANAYSIS\fire


In [5]:
import pandas as pd

df = pd.read_csv(DATA_DIR/'fire.csv')
df.head()

,시도,화재유형,발화요인대분류,발화요인소분류,장소대분류,인명피해_발생여부,발생_월,발생_시간,발생_요일
0,전북특별자치도,"건축,구조물",부주의,기타(부주의),생활서비스,0,1,0,2
1,서울특별시,기타(쓰레기 화재등),부주의,담배꽁초,기타,0,1,0,2
2,경상북도,기타(쓰레기 화재등),부주의,담배꽁초,기타,0,1,0,2
3,충청북도,"자동차,철도차량",기계적 요인,"과열, 과부하","자동차,철도차량",0,1,0,2
4,부산광역시,"자동차,철도차량",미상,미상,"자동차,철도차량",0,1,0,2


In [6]:
def season(month):
    if month in (3, 4, 5):
        return '봄'
    elif month in (6, 7, 8):
        return '여름'
    elif month in (9, 10, 11):
        return '가을'
    else:
        return '겨울'

In [10]:
def time(hour):
    if 0 <= hour <= 6:
        return '새벽'
    elif 7 <= hour <= 12:
        return '오전'
    elif 13 <= hour <= 17:
        return '오후'
    else:
        return '저녁'

In [7]:
df['계절'] = df['발생_월'].apply(season)

In [12]:
df['시간'] = df['발생_시간'].apply(time)

In [13]:
df.head()

,시도,화재유형,발화요인대분류,발화요인소분류,장소대분류,인명피해_발생여부,발생_월,발생_시간,발생_요일,계절,시간
0,전북특별자치도,"건축,구조물",부주의,기타(부주의),생활서비스,0,1,0,2,겨울,새벽
1,서울특별시,기타(쓰레기 화재등),부주의,담배꽁초,기타,0,1,0,2,겨울,새벽
2,경상북도,기타(쓰레기 화재등),부주의,담배꽁초,기타,0,1,0,2,겨울,새벽
3,충청북도,"자동차,철도차량",기계적 요인,"과열, 과부하","자동차,철도차량",0,1,0,2,겨울,새벽
4,부산광역시,"자동차,철도차량",미상,미상,"자동차,철도차량",0,1,0,2,겨울,새벽


In [14]:
feature_cols = ['발화요인소분류','발화요인대분류','계절','시간','발생_요일']
target_col = ['인명피해_발생여부']

X = df[feature_cols]
y = df[target_col]

X = X.astype(str)

X.head()

,발화요인소분류,발화요인대분류,계절,시간,발생_요일
0,기타(부주의),부주의,겨울,새벽,2
1,담배꽁초,부주의,겨울,새벽,2
2,담배꽁초,부주의,겨울,새벽,2
3,"과열, 과부하",기계적 요인,겨울,새벽,2
4,미상,미상,겨울,새벽,2


In [15]:
X_encode = pd.get_dummies(X)
oh = OneHotEncoder()

In [16]:
X_encode_train, X_encode_valid, y_train, y_valid = train_test_split(
    X_encode, y, test_size = 0.25, stratify =y, random_state=42
)
X_encode_train.shape, X_encode_valid.shape, y_train.shape, y_valid.shape

((143632, 76), (47878, 76), (143632, 1), (47878, 1))

In [17]:
model = make_pipeline(
    LogisticRegression(max_iter=1000)
)
model.fit(X_encode_train, y_train)

UnicodeDecodeError: 'cp949' codec can't decode byte 0xe2 in position 2950: illegal multibyte sequence

UnicodeDecodeError: 'cp949' codec can't decode byte 0xe2 in position 2950: illegal multibyte sequence

Pipeline(steps=[('logisticregression', LogisticRegression(max_iter=1000))])

In [22]:
# 1. 파이프라인 안에서 학습이 완료된 로지스틱 회귀 모델 꺼내기
lr_model = model.named_steps['logisticregression']

# 2. 61개 변수 이름과 각각의 위험도(Coef, 기울기) 매칭하기
coef_df = pd.DataFrame({
    '특징(시간/요일/원인)': X_encode.columns,
    '위험도(Coef)': lr_model.coef_[0]
})

# 3. 위험도가 가장 높은 순서대로(큰 플러스 값) 정렬하기
coef_df = coef_df.sort_values(by='위험도(Coef)', ascending=False)

# 4. 화면에 출력하기
coef_df


,특징(시간/요일/원인),위험도(Coef)
6,발화요인대분류_가스누출(폭발),2.135562
11,발화요인대분류_방화,1.226990
12,발화요인대분류_방화의심,1.206272
3,"화재유형_위험물,가스제조소등",0.912639
2,"화재유형_선박,항공기",0.572794
...,...,...
16,발화요인대분류_제품결함,-0.728277
15,발화요인대분류_전기적 요인,-1.188373
8,발화요인대분류_기계적 요인,-1.270487
1,화재유형_기타(쓰레기 화재등),-1.562760
